# Project 1: can a deep image network spot neighbourhoods that will beat their metro?

This notebook trains a convolutional network on the aerial photo of each ZIP code to predict how far
its home values will beat (or trail) its metro over the next 12 months. It uses the same approach as
Homework 7: a published network (MobileNetV2, trained on ImageNet) is borrowed, first frozen and then
partly retrained.

**The rules that keep the test honest.** The file you upload holds outcomes for target years 2014-2018
only. The network trains on 2014-2017 and uses 2018 to decide when to stop. It never sees an outcome
from 2019 (our validation year) or later. It does score every ZIP code for every year up to 2022, so the
scores are ready when the locked years are opened, but those scores use pictures only.

**Before you start:** `Runtime` > `Change runtime type` > `T4 GPU`. Then `Runtime` > `Run all`.
It takes about 30-40 minutes, most of it downloading and reading the 35,000 photos.

In [ ]:
import os, re, io, json, time, glob
import numpy as np, pandas as pd
from multiprocessing import Pool
from PIL import Image
import keras
from keras import layers
TEST_MODE = os.environ.get("CNN_TEST_MODE") == "1"     # used only to test this notebook offline
SEED = 15071
SIZE = 96 if TEST_MODE else 128           # pixels per side fed to the network
TRAIN_TARGET_YEARS = [2014, 2015, 2016, 2017]
STOP_TARGET_YEAR = 2018                   # used only to decide when to stop training
SCORE_ROW_YEARS = list(range(2013, 2023)) # every December from 2013 to 2022 gets a picture score
keras.utils.set_random_seed(SEED)
print("Devices:", keras.distribution.list_devices())
if not TEST_MODE and not keras.distribution.list_devices("gpu"):
    print("No GPU found. Switch the runtime to a T4 GPU, or training will take hours.")

## 1. Upload the training outcomes
Upload `cnn_targets.csv` when the button appears. One row per ZIP code and December (`row_year`), with
the growth versus metro over the following 12 months (`excess_growth_12m`), for target years 2014-2018.

In [ ]:
if TEST_MODE:
    targets = pd.read_csv("cnn_targets.csv")
else:
    from google.colab import files
    uploaded = files.upload()
    targets = pd.read_csv(io.BytesIO(uploaded["cnn_targets.csv"]))
assert targets.target_year.max() <= STOP_TARGET_YEAR, "file must not contain outcomes after 2018"
print(f"{len(targets):,} outcomes, target years {targets.target_year.min()}-{targets.target_year.max()}, "
      f"{targets.zipcode.nunique():,} ZIP codes")

## 2. Download the aerial photos from Kaggle (about 12 GB, 5-10 minutes)
If it asks for a login, paste your Kaggle username and API key
(kaggle.com > your picture > Settings > API > Create New Token).

In [ ]:
if TEST_MODE:
    root = "synthetic_images"
else:
    import kagglehub
    try:
        root = kagglehub.dataset_download("shengkunwang/housets-dataset")
    except Exception as e:
        print("Login needed:", e); kagglehub.login()
        root = kagglehub.dataset_download("shengkunwang/housets-dataset")
NAME = re.compile(r"^(\d+)_(\d{4})\.png$")
index = pd.DataFrame([{"zipcode": int(m.group(1)), "image_year": int(m.group(2)), "path": p}
                      for p in glob.glob(os.path.join(root, "**", "*.png"), recursive=True)
                      if (m := NAME.match(os.path.basename(p)))])
index = index[index.zipcode.isin(targets.zipcode.unique())].sort_values(["zipcode", "image_year"]).reset_index(drop=True)
print(f"{len(index):,} photos of {index.zipcode.nunique():,} ZIP codes, years {index.image_year.min()}-{index.image_year.max()}")

## 3. Read every photo once
Washington DC photos come in a different shape, so they are padded to a square rather than stretched.
A photo that covers less than half of its ZIP code's fullest photo was clipped by the camera tile; it is
skipped and the previous good photo is used instead.

In [ ]:
def read_photo(path):
    img = Image.open(path).convert("RGB")
    if img.size != (512, 512):
        img.thumbnail((512, 512), Image.BILINEAR)
        canvas = Image.new("RGB", (512, 512)); canvas.paste(img, ((512 - img.size[0]) // 2, (512 - img.size[1]) // 2)); img = canvas
    rgb = np.asarray(img)
    coverage = float((rgb > 8).any(axis=2).mean())          # share of the frame inside the ZIP code
    return np.asarray(img.resize((SIZE, SIZE), Image.BILINEAR), dtype=np.uint8), coverage

t0 = time.time()
CACHE = f"photos_{SIZE}.npy"                               # saved so a rerun skips this step
if os.path.exists(CACHE) and os.path.exists("coverage.npy") and len(np.load("coverage.npy")) == len(index):
    photos = np.load(CACHE, mmap_mode="r"); index["coverage"] = np.load("coverage.npy")
else:
    photos = np.empty((len(index), SIZE, SIZE, 3), dtype=np.uint8)   # one copy only, filled in place
    coverage = np.empty(len(index))
    with Pool(os.cpu_count()) as pool:
        for i, (img, cov) in enumerate(pool.imap(read_photo, index.path.tolist(), chunksize=64)):
            photos[i], coverage[i] = img, cov
    np.save(CACHE, photos); np.save("coverage.npy", coverage); index["coverage"] = coverage
index["good"] = index.coverage >= 0.5 * index.groupby("zipcode").coverage.transform("max")
print(f"read {len(photos):,} photos in {time.time() - t0:.0f}s; {int((~index.good).sum()):,} clipped photos skipped")

## 4. Match each ZIP code and December to its latest photo taken on or before that December

In [ ]:
good = index[index.good].reset_index().rename(columns={"index": "photo"})
rows = pd.DataFrame([(z, y) for z in targets.zipcode.unique() for y in SCORE_ROW_YEARS], columns=["zipcode", "row_year"])
rows = pd.merge_asof(rows.sort_values("row_year"), good[["zipcode", "image_year", "photo"]].sort_values("image_year"),
                     left_on="row_year", right_on="image_year", by="zipcode", direction="backward").dropna(subset=["photo"])
rows["photo"] = rows.photo.astype(int)
rows = rows.merge(targets[["zipcode", "row_year", "city", "target_year", "excess_growth_12m"]], on=["zipcode", "row_year"], how="left")
rows["city"] = rows.city.fillna(rows.zipcode.map(targets.drop_duplicates("zipcode").set_index("zipcode").city))
train = rows[rows.target_year.isin(TRAIN_TARGET_YEARS)]; stop = rows[rows.target_year == STOP_TARGET_YEAR]
SD = float(train.excess_growth_12m.std())
print(f"training rows {len(train):,}; stopping rows {len(stop):,}; rows to score {len(rows):,}")

## 5. The network
Seen from directly above, a neighbourhood is the same neighbourhood mirrored or turned round, so flipped
photos are free, honest extra data. The borrowed network reads the photo; a small head on top turns what
it sees into a forecast. The loss is Huber, which is less thrown by a few extreme ZIP codes.

In [ ]:
def build(unfreeze_from=None, learning_rate=1e-3):
    keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
    base = keras.applications.MobileNetV2(input_shape=(SIZE, SIZE, 3), include_top=False,
                                          weights=None if TEST_MODE else "imagenet")
    base.trainable = unfreeze_from is not None
    if unfreeze_from is not None:
        names = [l.name for l in base.layers]; cut = names.index(unfreeze_from)
        for l in base.layers[:cut]: l.trainable = False
        for l in base.layers: 
            if isinstance(l, layers.BatchNormalization): l.trainable = False
    photo = layers.Input(shape=(SIZE, SIZE, 3))
    x = layers.RandomFlip("horizontal_and_vertical")(photo)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)              # MobileNetV2 expects values in [-1, 1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(photo, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate), loss=keras.losses.Huber())
    return model

class Batches(keras.utils.PyDataset):
    """Feeds the network a few photos at a time, so the photos are never copied in full."""
    def __init__(self, photo_ids, y=None, shuffle=False):
        super().__init__()
        self.ids, self.y, self.shuffle = np.asarray(photo_ids), y, shuffle
        self.order, self.rng = np.arange(len(self.ids)), np.random.default_rng(SEED)
        self.on_epoch_end()
    def __len__(self):
        return int(np.ceil(len(self.ids) / 64))
    def __getitem__(self, i):
        b = self.order[i * 64:(i + 1) * 64]
        x = np.asarray(photos[np.sort(self.ids[b])][np.argsort(np.argsort(self.ids[b]))])
        return x if self.y is None else (x, self.y[b])
    def on_epoch_end(self):
        if self.shuffle: self.rng.shuffle(self.order)

def predict(model, photo_ids):
    return model.predict(Batches(photo_ids), verbose=0).ravel()

def rank_corr(df, pred):
    d = df.assign(p=pred)
    r = [g.p.corr(g.excess_growth_12m, method="spearman") for _, g in d.groupby("city") if len(g) >= 10]
    r = [v for v in r if not np.isnan(v)]
    return float(np.mean(r)) if r else -1.0

def fit(model, epochs, patience):
    t0 = time.time()
    print(f"trainable weights: {sum(int(np.prod(w.shape)) for w in model.trainable_weights):,}")
    h = model.fit(Batches(train.photo.values, train.excess_growth_12m.values / SD, shuffle=True), epochs=epochs,
                  validation_data=Batches(stop.photo.values, stop.excess_growth_12m.values / SD), verbose=2,
                  callbacks=[keras.callbacks.EarlyStopping(patience=patience, restore_best_weights=True)])
    p = predict(model, stop.photo.values)
    return {"epochs_run": len(h.history["loss"]), "best_stop_loss": float(min(h.history["val_loss"])),
            "stop_year_rank_corr": rank_corr(stop, p), "seconds": round(time.time() - t0)}

## 6. Train: frozen first, then retrain the top of the network

In [ ]:
log = {}
frozen = build()
log["frozen"] = fit(frozen, epochs=2 if TEST_MODE else 25, patience=4)
print("frozen:", log["frozen"])
tuned = build(unfreeze_from="block_13_expand", learning_rate=1e-5)
tuned.set_weights(frozen.get_weights())                      # start from the trained head
log["fine_tuned"] = fit(tuned, epochs=2 if TEST_MODE else 12, patience=3)
print("fine-tuned:", log["fine_tuned"])
chosen = "fine_tuned" if log["fine_tuned"]["stop_year_rank_corr"] > log["frozen"]["stop_year_rank_corr"] else "frozen"
log.update({"chosen": chosen, "image_size": SIZE, "train_target_years": TRAIN_TARGET_YEARS, "stop_target_year": STOP_TARGET_YEAR,
            "training_rows": int(len(train)), "target_sd": SD, "photos_read": int(len(photos)), "clipped_photos_skipped": int((~index.good).sum())})
print("chosen on 2018:", chosen)

## 7. Score every ZIP code and December, and download the results
Two files download: `image_scores.csv` (one picture-based score per ZIP code and December, 2013-2022)
and `image_training_log.json`. Put both in the `ai-edge/Project 1` folder on your computer.

In [ ]:
for name, model in [("frozen", frozen), ("fine_tuned", tuned)]:
    rows[f"score_{name}"] = predict(model, rows.photo.values) * SD
rows["image_score"] = rows[f"score_{chosen}"]
rows["used_in_training"] = rows.target_year.isin(TRAIN_TARGET_YEARS + [STOP_TARGET_YEAR])
scores = rows[["zipcode", "row_year", "image_year", "image_score", "score_frozen", "score_fine_tuned", "used_in_training"]]
scores.sort_values(["zipcode", "row_year"]).to_csv("image_scores.csv", index=False)
json.dump(log, open("image_training_log.json", "w"), indent=1)
print(scores.groupby("row_year").size().to_dict())
if not TEST_MODE:
    files.download("image_scores.csv"); files.download("image_training_log.json")